# DALIL : classifieur de famille de pièce par encodeur multilingue affiné (variante B)

Paramètres Kaggle : **GPU T4**, **Internet activé**. Sortie : dossier ONNX à copier dans `ml/artifacts/` puis à évaluer avec `python ml/evaluate.py --model B=ml/artifacts/<dossier>`.

Configurations (section 6.3) : `fr` (textes traduits en français), `en+fr` (textes anglais d'origine + français : transfert multilingue sans traduction) et `set` : le jeu commun `data/processed/train_set/` (`python ml/build_training_set.py`), le même que la variante A, avec un poids par exemple selon la confiance de l'étiquette (L'Argus, NHTSA…).

In [ ]:
import glob, json, os, random, shutil, time
from collections import Counter
# Le jeu de données « dalil-pipeline » est monté sous /kaggle/input (chemin variable selon Kaggle) :
# on cherche le dossier qui contient backend/ et data/.
INPUT = next(root for root, dirs, _ in os.walk("/kaggle/input") if "backend" in dirs and "data" in dirs)
REPO = "/kaggle/working/dalil"
if not os.path.exists(REPO):
    shutil.copytree(INPUT, REPO)
os.chdir(REPO)
print(sorted(os.listdir(REPO)))

In [ ]:
!pip install -q -U "transformers>=4.48" accelerate onnx onnxruntime onnxscript

In [ ]:
MODEL_ID = "jhu-clsp/mmBERT-base"   # à comparer : "jhu-clsp/mmBERT-base", "almanach/camembert-base", "almanach/camembertav2-base"
CONFIG = "en+fr"    # "set" (jeu commun pondéré), "fr" (textes traduits) ou "en+fr" (transfert depuis l'anglais)
MAX_LEN, EPOCHS, LR, BATCH, SEED, MIN_PER_CLASS = 256, 3, 3e-5, 32, 13, 5
SET_DIR, TARGET = "data/processed/train_set", "part"   # jeu commun et cible (« part » ou « family »)
NAME = f"{MODEL_ID.split('/')[-1]}_{CONFIG.replace('+', '-')}"
OUT = f"/kaggle/working/{NAME}"
os.makedirs(OUT, exist_ok=True)

In [ ]:
def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

faults = json.load(open("data/seed/faults.json", encoding="utf-8"))["faults"]
f2p = {f["slug"]: max(f["parts"].items(), key=lambda x: x[1])[0] for f in faults}
examples = [{"text": r["text"], "label": f2p[r["fault"]], "src": "seed", "weight": 1.0}
            for r in load_jsonl("data/seed/diagnosis_examples_fr.jsonl")]
for p in ([] if CONFIG == "set" else sorted(glob.glob("data/processed/train/*.jsonl"))):
    for r in load_jsonl(p):
        label = r.get("category") or r.get("family")
        if not label:
            continue
        if CONFIG == "fr":
            text = r["text"] if r["lang"] == "fr" else r.get("text_fr")
        else:
            text = r["text"] if r["lang"] in ("en", "fr") else r.get("text_fr")
        if text:
            examples.append({"text": text[:2000], "label": label, "src": r["source"], "weight": 1.0})
if CONFIG == "set":   # même découpage et mêmes poids que la variante A (train_baseline.py --set)
    print(json.load(open(f"{SET_DIR}/stats.json", encoding="utf-8")))
    train = [{**r, "src": r["source"]} for r in load_jsonl(f"{SET_DIR}/train.jsonl")]
    val = [{**r, "src": r["source"]} for r in load_jsonl(f"{SET_DIR}/val.jsonl")]
    counts = Counter(e["label"] for e in train)
    train = [e for e in train if counts[e["label"]] >= MIN_PER_CLASS]
    labels = sorted({e["label"] for e in train})
    val = [e for e in val if e["label"] in labels]
    examples = train + val
else:
    counts = Counter(e["label"] for e in examples)
    examples = [e for e in examples if counts[e["label"]] >= MIN_PER_CLASS]
    labels = sorted({e["label"] for e in examples})
    random.Random(SEED).shuffle(examples)
    n_val = max(50, len(examples) // 10)
    val, train = examples[:n_val], examples[n_val:]
lid = {l: i for i, l in enumerate(labels)}
dev = [d for d in load_jsonl("data/test/dev_fr_synthetique.jsonl") if d.get("text") and d.get("expect")]
if TARGET == "family":   # attendus du jeu dev = pièces -> familles
    _cats = json.load(open("data/seed/part_categories.json", encoding="utf-8"))["families"]
    _p2f = {c["slug"]: f["slug"] for f in _cats for c in f.get("children", [])} | {f["slug"]: f["slug"] for f in _cats}
    dev = [{**d, "expect": sorted({_p2f.get(x, x) for x in d["expect"]})} for d in dev]
print(f"{len(train)} entraînement, {len(val)} validation, {len(dev)} dev FR, {len(labels)} classes")
print(Counter(e["src"] for e in examples))

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=lid).to("cuda")

def collate(batch):
    enc = tok([b["text"] for b in batch], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor([lid[b["label"]] for b in batch])
    enc["weights"] = torch.tensor([float(b.get("weight", 1.0)) for b in batch])
    return enc

@torch.no_grad()
def predict(texts, bs=64):
    model.eval()
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(model(**enc).logits.float().softmax(-1).cpu())
    model.train()
    return torch.cat(out)

def topk(probs, gold_sets, k):
    idx = probs.topk(k, dim=-1).indices.tolist()
    return round(sum(any(labels[j] in g for j in row) for row, g in zip(idx, gold_sets)) / len(gold_sets), 3)

def evaluate():
    pv = predict([e["text"] for e in val])
    pd = predict([d["text"] for d in dev])
    gv, gd = [{e["label"]} for e in val], [set(d["expect"]) for d in dev]
    return {"val_top1": topk(pv, gv, 1), "val_top3": topk(pv, gv, 3), "dev_top1": topk(pd, gd, 1), "dev_top3": topk(pd, gd, 3)}

dl = DataLoader(train, batch_size=BATCH, shuffle=True, collate_fn=collate)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, int(0.06 * len(dl) * EPOCHS), len(dl) * EPOCHS)
scaler = torch.amp.GradScaler("cuda")
ckpt, start, history = f"{OUT}/checkpoint.pt", 0, []
if os.path.exists(ckpt):   # reprise après une session Kaggle interrompue (sauvegarde à chaque époque)
    state = torch.load(ckpt, weights_only=False)
    model.load_state_dict(state["model"]); opt.load_state_dict(state["opt"]); sched.load_state_dict(state["sched"])
    start, history = state["epoch"] + 1, state["history"]
for epoch in range(start, EPOCHS):
    t0 = time.time()
    for step, batch in enumerate(dl):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        y, w = batch.pop("labels"), batch.pop("weights")
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**batch).logits
        # perte pondérée : une étiquette tirée des réponses du forum compte moins qu'une solution confirmée
        loss = (torch.nn.functional.cross_entropy(logits.float(), y, reduction="none") * w).sum() / w.sum()
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        if step % 100 == 0:
            print(f"époque {epoch} pas {step}/{len(dl)} perte {loss.item():.3f}")
    history.append({"epoch": epoch, **evaluate(), "minutes": round((time.time() - t0) / 60, 1)})
    print(history[-1])
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "epoch": epoch, "history": history}, ckpt)

In [ ]:
# Export ONNX : le portable l'exécute avec onnxruntime, sans PyTorch
model.save_pretrained(f"{OUT}/hf"); tok.save_pretrained(f"{OUT}/hf")
export = AutoModelForSequenceClassification.from_pretrained(f"{OUT}/hf", attn_implementation="eager").eval()

class Logits(torch.nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits

onnx_dir = f"{OUT}/onnx"
os.makedirs(onnx_dir, exist_ok=True)
sample = tok(["le volant tremble quand je freine"], return_tensors="pt")
torch.onnx.export(Logits(export), (sample["input_ids"], sample["attention_mask"]), f"{onnx_dir}/model.onnx",
                  input_names=["input_ids", "attention_mask"], output_names=["logits"],
                  dynamic_axes={"input_ids": {0: "b", 1: "t"}, "attention_mask": {0: "b", 1: "t"}, "logits": {0: "b"}},
                  opset_version=17, dynamo=False)   # exportateur classique : axes dynamiques fiables
tok.save_pretrained(onnx_dir)   # écrit tokenizer.json
json.dump({"target": TARGET, "labels": labels, "version": f"{NAME}-{time.strftime('%Y%m%d')}", "max_length": MAX_LEN,
           "history": history, "model_id": MODEL_ID, "config": CONFIG},
          open(f"{onnx_dir}/labels.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

# Vérification : mêmes sorties en ONNX et en PyTorch
import numpy as np, onnxruntime as ort
sess = ort.InferenceSession(f"{onnx_dir}/model.onnx", providers=["CPUExecutionProvider"])
o = sess.run(None, {"input_ids": sample["input_ids"].numpy(), "attention_mask": sample["attention_mask"].numpy()})[0]
with torch.no_grad():
    t = export(**sample).logits.numpy()
print("écart max ONNX / PyTorch :", float(np.abs(o - t).max()))

# Quantification int8 : modèle ~4 fois plus léger et plus rapide sur le processeur du portable.
# On ne la garde que si la précision sur le jeu dev ne baisse pas de plus d'un point.
from onnxruntime.quantization import QuantType, quantize_dynamic
quantize_dynamic(f"{onnx_dir}/model.onnx", f"{OUT}/model.int8.onnx", weight_type=QuantType.QInt8)

def ort_dev_scores(path):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    rows = []
    for d in dev:
        enc = tok([d["text"]], truncation=True, max_length=MAX_LEN, return_tensors="np")
        logits = sess.run(None, {"input_ids": enc["input_ids"].astype(np.int64), "attention_mask": enc["attention_mask"].astype(np.int64)})[0][0]
        rows.append(torch.tensor(logits).softmax(-1))
    probs = torch.stack(rows)
    gd = [set(d["expect"]) for d in dev]
    return {"dev_top1": topk(probs, gd, 1), "dev_top3": topk(probs, gd, 3)}

fp32, int8 = ort_dev_scores(f"{onnx_dir}/model.onnx"), ort_dev_scores(f"{OUT}/model.int8.onnx")
print("fp32", fp32, "int8", int8)
meta = json.load(open(f"{onnx_dir}/labels.json", encoding="utf-8"))
meta.update({"onnx_fp32": fp32, "onnx_int8": int8})
if int8["dev_top1"] >= fp32["dev_top1"] - 0.01:
    shutil.move(f"{OUT}/model.int8.onnx", f"{onnx_dir}/model.onnx")
    meta["quantization"] = "int8 dynamique"
json.dump(meta, open(f"{onnx_dir}/labels.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("taille du modèle :", round(os.path.getsize(f"{onnx_dir}/model.onnx") / 1e6), "Mo", meta.get("quantization", "fp32"))
shutil.make_archive(f"/kaggle/working/{NAME}_onnx", "zip", onnx_dir)
print(history)